# Lecture 0 · Probability + MLE in code

Watch every loss in DL pop out of one principle · Maximum Likelihood Estimation.

We'll:
1. Sample from the three core distributions (Bernoulli, Categorical, Normal).
2. Visualize log-likelihood as a function of one parameter.
3. Derive MSE for linear regression by maximizing log-likelihood.
4. Derive cross-entropy for logistic regression by maximizing log-likelihood.
5. See log-sum-exp save us from overflow.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
np.random.seed(0); torch.manual_seed(0)

## 1 · Bernoulli sampling and pmf

In [ ]:
p = 0.7
samples = np.random.rand(1000) < p
print(f'fraction of 1s · {samples.mean():.3f}  (true p = {p})')

# pmf
print(f'P(Y=1) · {p} · P(Y=0) · {1-p}')
# joint of 3 independent flips
print(f'P(1, 0, 1) · {p * (1-p) * p:.4f}')

## 2 · Categorical sampling via inverse CDF

In [ ]:
pi = np.array([0.1, 0.3, 0.4, 0.2])
cdf = np.cumsum(pi)
print(f'cumulative · {cdf}')

# sample 5000 times
u = np.random.rand(5000)
samples = np.searchsorted(cdf, u)
for k in range(4):
    print(f'class {k} · empirical {(samples == k).mean():.3f} · true {pi[k]:.3f}')

## 3 · Normal sampling and density

Sample from $\mathcal{N}(0, 1)$ via Box-Muller (or just `torch.randn`).

In [ ]:
y = np.linspace(-5, 5, 200)
for mu, sigma in [(0, 1), (0, 2), (2, 0.5)]:
    pdf = np.exp(-0.5 * ((y - mu) / sigma)**2) / (sigma * np.sqrt(2 * np.pi))
    plt.plot(y, pdf, label=f'N({mu}, {sigma}^2)')
plt.legend(); plt.title('Normal densities'); plt.show()

## 4 · MSE pops out of MLE for linear regression

Generate data · $y = 2x + 1 + \epsilon$ where $\epsilon \sim \mathcal{N}(0, 0.5^2)$.

In [ ]:
x = np.linspace(-2, 2, 100)
y = 2 * x + 1 + 0.5 * np.random.randn(100)

# Sweep over candidate slopes w with bias fixed at 1
ws = np.linspace(0, 4, 200)
log_likelihoods = []
mses = []
sigma = 0.5
for w in ws:
    pred = w * x + 1
    log_lik = -0.5 * np.sum((y - pred)**2) / sigma**2 - 100 * 0.5 * np.log(2 * np.pi * sigma**2)
    mse = np.mean((y - pred)**2)
    log_likelihoods.append(log_lik); mses.append(mse)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(ws, log_likelihoods); axes[0].axvline(2, c='r', ls='--', label='true w=2')
axes[0].set_title('Log-likelihood'); axes[0].legend(); axes[0].set_xlabel('w')
axes[1].plot(ws, mses); axes[1].axvline(2, c='r', ls='--', label='true w=2')
axes[1].set_title('MSE (negative log-lik · constants)'); axes[1].legend(); axes[1].set_xlabel('w')
plt.show()

print(f'argmax(log-lik) · w = {ws[np.argmax(log_likelihoods)]:.2f}')
print(f'argmin(MSE)     · w = {ws[np.argmin(mses)]:.2f}')
print('Same answer · MSE IS MLE under Gaussian noise.')

## 5 · Cross-entropy pops out of MLE for logistic regression

Generate binary data · two Gaussians, label by which one.

In [ ]:
x_pos = np.random.randn(100, 1) + 1.0
x_neg = np.random.randn(100, 1) - 1.0
X = np.vstack([x_pos, x_neg])
y = np.array([1]*100 + [0]*100)

def sigmoid(z): return 1 / (1 + np.exp(-z))

ws = np.linspace(-1, 5, 200)
log_liks = []
ces = []
for w in ws:
    p = sigmoid(w * X[:,0])
    log_lik = np.sum(y * np.log(p + 1e-12) + (1-y) * np.log(1-p + 1e-12))
    ce = -log_lik / len(y)  # mean BCE
    log_liks.append(log_lik); ces.append(ce)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(ws, log_liks); axes[0].set_title('Log-likelihood'); axes[0].set_xlabel('w')
axes[1].plot(ws, ces); axes[1].set_title('Binary cross-entropy'); axes[1].set_xlabel('w')
plt.show()

print(f'argmax(log-lik) · w = {ws[np.argmax(log_liks)]:.2f}')
print(f'argmin(BCE)     · w = {ws[np.argmin(ces)]:.2f}')

## 6 · Numerical stability · log-sum-exp

In [ ]:
logits = torch.tensor([1000.0, 999.0, 998.0])

# Naive softmax · overflows
try:
    naive = torch.exp(logits) / torch.exp(logits).sum()
    print('naive softmax', naive)
except Exception as e:
    print('naive failed', e)

# Stable softmax
shifted = logits - logits.max()
stable = torch.exp(shifted) / torch.exp(shifted).sum()
print('stable softmax', stable)

# PyTorch builtins
print('F.softmax', F.softmax(logits, dim=-1))
print('F.log_softmax', F.log_softmax(logits, dim=-1))

## 7 · Cross-entropy gradient is `prediction - truth`

In [ ]:
logits = torch.tensor([2.0, 1.0, 0.1], requires_grad=True)
target = torch.tensor([0])  # class 0 is correct
loss = F.cross_entropy(logits.unsqueeze(0), target)
loss.backward()

softmax = F.softmax(logits, dim=-1)
print('softmax · ', softmax.detach())
print('one-hot · ', torch.eye(3)[0])
print('softmax - onehot · ', (softmax.detach() - torch.eye(3)[0]))
print('actual gradient · ', logits.grad)